In [1]:
# root file producer for the 16x16
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd
from openpyxl import Workbook
import pytz

Welcome to JupyROOT 6.30/04


In [ ]:
file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/HPK/PRE-SERIE_HPK_Vendor_16x16_IV.root", "RECREATE")
#file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/HPK/pippo.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
sensor = array('i', [0])
row = array('i', [0])
column = array('i', [0])
temperature = array('f', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("sensor", sensor,'sensor/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')
tree.Branch("temperature", temperature, 'temperature/F')


V.reserve(1000)
IBACK.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)

csv_files = glob.glob("/Users/icosivi/Desktop/PRE-SERIE/HPK/on-wafer_data/*.xlsx")

wb = pd.read_excel('/Users/icosivi/Desktop/PRE-SERIE/HPK/HPK_sensor-col-row.xlsx')

nevent = 0

for csv in csv_files:
  #print(csv)
  df = pd.read_excel(csv, sheet_name='IV', header=None)
  nome_senza_ext = os.path.splitext(csv)[0]
  wafer[0] = int(nome_senza_ext.split('-')[-1])

  for i in range(24):
      V.clear()
      IBACK.clear()
      
      event[0] = nevent
      sensor[0] = int(df.iat[0,i+1])
      c, r = wb.loc[wb['Sensor'] == int(i+1), ['Column', 'Row']].values[0]
      column[0] = int(c)
      row[0] = int(r)
      
      I_list = df.iloc[3:,i+1].dropna().tolist()
      if I_list:
        temperature[0] = float(df.iat[1,i+1])
        subset = df.iloc[3:, [0, i+1]].dropna() 
        for _, rr in subset.iterrows():  
          IBACK.push_back( float(rr.iloc[1]) )
          V.push_back( float(rr.iloc[0]) )
        
      tree.Fill()
      nevent += 1
      
      
      #V_list = df[3:,0].dropna().tolist()
      #for v in V_list:
      #  V.push_back( float(v) )

tree.Write()
file.Write()
file.Close()
      
      
      
      
      
  

In [2]:
def to_binary(number, bit_length):
    """
    Converte un numero decimale in una stringa binaria di lunghezza fissa.
    
    Argomenti:
        number (int): Il numero decimale da convertire.
        bit_length (int): Il numero di bit desiderati in output.
        
    Ritorna:
        str: La rappresentazione binaria invertita (Little Endian) del numero.
    """
    # Converte il numero in binario standard (es: "0b1"), 
    # rimuove il prefisso '0b' e aggiunge zeri a sinistra fino alla lunghezza desiderata
    binary_standard = bin(number)[2:].zfill(bit_length)
    
    # Se il numero originale era più lungo del bit_length, tronchiamo per sicurezza
    binary_fixed = binary_standard[-bit_length:]
    
    # Invertiamo la stringa come richiesto dall'esempio (1 -> 10000 invece di 00001)
    return binary_fixed[::-1]


def to_decimal(binary_str):
    """
    Converte una stringa binaria invertita (Little Endian) nel suo valore decimale.
    Esempio: "10000" -> 1
    """
    # Inverte la stringa per riportare il bit meno significativo (LSB) alla fine
    binary_standard = binary_str[::-1]
    
    # Converte la stringa binaria standard in un intero decimale
    return int(binary_standard, 2)

In [4]:
# producer of the xls to register components on the database for the 16x16
xl_filename="HPK_16x16_PRE-SERIES"
wb = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

file_qa = root.TFile.Open("/Users/icosivi/Desktop/PRE-SERIE/HPK/PRE-SERIE_HPK_Vendor_16x16_IV.root")
tree_qa = file_qa.Get("Tree")

for j,event in enumerate(tree_qa):
    
    vendor_bit = 0
    wafer_bit = to_binary(event.wafer,14)
    sensor_bit = to_binary(event.sensor,5)
    wws["A%i" %(j+2)] = 'PRE'+str(vendor_bit)+str(wafer_bit)+str(sensor_bit)
    
    wws["B%i" %(j+2)] = 'HPK'
    wws["C%i" %(j+2)] = None
    wws["D%i" %(j+2)] = event.wafer
    wws["E%i" %(j+2)] = '16x16'
    
    #Column, Row = wb.loc[df['Sensor'] == sensor_number, ['Column', 'Row']].values[0]

    wws["F%i" %(j+2)] = event.row
    wws["G%i" %(j+2)] = event.column
    wws["H%i" %(j+2)] = event.sensor

save_path = '/Users/icosivi/Desktop/PRE-SERIE/HPK/'
wb.save(save_path+xl_filename+".xlsx")